This script was used to transform the Global Mangrove Aboveground Biomass and Canopy Height  dataset from GeoTIFF to Cloud Optimized GeoTIFF (COG) format for display in the Greenhouse Gas (GHG) Center.

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
from rasterio.warp import calculate_default_transform, reproject, Resampling
import botocore
from pathlib import Path
from dotenv import load_dotenv

In [6]:
config = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : "ghgc-data-store-dev",
    "raw_data_prefix": "coastal-observatory/data",
    "cog_data_bucket": "ghgc-data-store-dev",
    "cog_data_prefix": "transformed_cogs/CMS_Global_Map_Mangrove_Canopy_Biomass",
    "local_output_dir": "output/cms-global-map-mangrove-biomass",  # Local directory to save COGs
    "transformation": {}
}

In [7]:
# Load environment variables from .env file
load_dotenv()

# AWS Credentials from environment variables
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
AWS_SESSION_TOKEN = os.getenv('AWS_SESSION_TOKEN')

Approach

1.) Read .tif files from S3 bucket
2.) Convert to COGs on local drive
3.) Move converted COGs to their final S3 location

In [8]:
session = boto3.session.Session()
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token = AWS_SESSION_TOKEN

)

bucket_name = config["cog_data_bucket"]
raw_data_bucket = config["raw_data_bucket"]
raw_data_prefix= config["raw_data_prefix"]

cog_data_bucket = config['cog_data_bucket']
cog_data_prefix= config["cog_data_prefix"]

fs = s3fs.S3FileSystem()

In [9]:
def get_all_s3_keys(bucket, model_name, ext):
    """Get a list of all keys in an S3 bucket."""
    keys = []

    kwargs = {"Bucket": bucket, "Prefix": f"{model_name}/"}
    while True:
        resp = s3_client.list_objects_v2(**kwargs)
        for obj in resp["Contents"]:
            if obj["Key"].endswith(ext) and "historical" not in obj["Key"]:
                keys.append(obj["Key"])

        try:
            kwargs["ContinuationToken"] = resp["NextContinuationToken"]
        except KeyError:
            break

    return keys

keys = get_all_s3_keys(raw_data_bucket, raw_data_prefix, ".tif")
keys

['coastal-observatory/data/Mangrove_agb_AndamanAndNicobar.tif',
 'coastal-observatory/data/Mangrove_agb_Angola.tif',
 'coastal-observatory/data/Mangrove_agb_Anguilla.tif',
 'coastal-observatory/data/Mangrove_agb_AntiguaAndBarbuda.tif',
 'coastal-observatory/data/Mangrove_agb_Aruba.tif',
 'coastal-observatory/data/Mangrove_agb_Australia.tif',
 'coastal-observatory/data/Mangrove_agb_Bahamas.tif',
 'coastal-observatory/data/Mangrove_agb_Bahrain.tif',
 'coastal-observatory/data/Mangrove_agb_Bangladesh.tif',
 'coastal-observatory/data/Mangrove_agb_Barbados.tif',
 'coastal-observatory/data/Mangrove_agb_Belize.tif',
 'coastal-observatory/data/Mangrove_agb_Benin.tif',
 'coastal-observatory/data/Mangrove_agb_Brazil.tif',
 'coastal-observatory/data/Mangrove_agb_BritishVirginIslands.tif',
 'coastal-observatory/data/Mangrove_agb_Brunei.tif',
 'coastal-observatory/data/Mangrove_agb_Cambodia.tif',
 'coastal-observatory/data/Mangrove_agb_Cameroon.tif',
 'coastal-observatory/data/Mangrove_agb_Carribea

In [10]:
def create_cog_filename(f):
    
    f = Path(f).stem
    # Example: "Mangrove_agb_AndamanAndNicobar.tif" -> "Mangrove_agb_AndamanAndNicobar_2000year.tif"
    # Example: "Mangrove_hmax95_Yemen.tif" -> "Mangrove_hmax95_Yemen_2000year.tif"
    
    # Simply append 2000year to the stem
    cog_filename = f"{f}_2000year.tif"
    return cog_filename

In [11]:
# Define COG profile for rasterio
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

In [32]:
def convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"
    
    # Create a temporary file for the downloaded S3 object
    temp_input_file = f"temp_{os.path.basename(name)}"

    try:
        # Download the file from S3 first
        print(f"[DOWNLOAD] Downloading {name} from S3...")
        s3_client.download_file(raw_data_bucket, name, temp_input_file)
        
        # Reproject using the local file
        print(f"[REPROJECT] {name} → {reproject_filename} (EPSG:4326)")
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds
            )
            kwargs = src.meta.copy()
            kwargs.update({
                "driver": "COG",                 # write a COG instead of plain GTiff
                "compress": "DEFLATE",           # or "LZW"
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height
            })

            with rasterio.open(f"{reproject_filename}", "w", **kwargs) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest,
                    wrapdateline=True
                )

        # 3) COGify & upload
        print(f"[COGIFY] {reproject_filename} → s3://{cog_data_bucket}/{s3_key}")
        ds = rxr.open_rasterio(reproject_filename)
        ds = ds.rename({"y": "lat", "x": "lon"})
        ds.rio.set_spatial_dims("lon", "lat", inplace=True)
        ds.rio.write_nodata(-9999, inplace=True)

        with tempfile.NamedTemporaryFile() as tmp:
            ds.rio.to_raster(tmp.name, **COG_PROFILE)
            
            # Upload to S3
            s3_client.upload_file(
                Filename = tmp.name, 
                Bucket = cog_data_bucket, 
                Key = s3_key)
            print(f"[SUCCESS] Uploaded to s3://{cog_data_bucket}/{s3_key}")
            
            # Save locally if output directory is specified
            if local_output_dir:
                os.makedirs(local_output_dir, exist_ok=True)
                local_path = os.path.join(local_output_dir, cog_filename)
                
                # Copy the COG file to local directory
                import shutil
                shutil.copy(tmp.name, local_path)
                print(f"[LOCAL SAVE] Saved COG to {local_path}")
            
    except Exception as e:
        print(f"[ERROR] Failed to process {name}: {str(e)}")
        raise
            
    finally:
        # Clean up temporary input file
        if os.path.exists(temp_input_file):
            os.remove(temp_input_file)
            print(f"[CLEANUP] removed temporary input file {temp_input_file}")
            
        # Clean up local intermediate
        if os.path.exists(reproject_filename):
            os.remove(reproject_filename)
            print(f"[CLEANUP] removed intermediate {reproject_filename}")

In [ ]:
# Initialize DataFrame to track processed files
files_processed = pd.DataFrame(columns=["file_name", "COGs_created"])

# Get local output directory from config
local_output_dir = config.get("local_output_dir")

# Create output directories
if local_output_dir:
    os.makedirs(local_output_dir, exist_ok=True)
    print(f"Local COGs will be saved to: {local_output_dir}")

# Process all files
for name in sorted(keys):
    cog_filename = create_cog_filename(name, start_str, end_str)
    print(f"\nProcessing: {name}")
    print(f"Output filename: {cog_filename}")
    
    # Process the file with local output directory
    convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir)
    
    # Add to tracking DataFrame
    files_processed = files_processed._append(
        {"file_name": name, "COGs_created": cog_filename},
        ignore_index=True,
    )
    print(f"Generated and saved COG: {cog_filename}")

print("\nDone generating COGs")
if local_output_dir:
    print(f"COGs saved locally to: {local_output_dir}")

Local COGs will be saved to: output/cms-global-map-mangrove

Processing: coastal-observatory/data/Mangrove_agb_AndamanAndNicobar.tif
Output filename: Mangrove_agb_AndamanAndNicobar_2000-01-01day_2009-12-31.tif
[DOWNLOAD] Downloading coastal-observatory/data/Mangrove_agb_AndamanAndNicobar.tif from S3...
[REPROJECT] coastal-observatory/data/Mangrove_agb_AndamanAndNicobar.tif → reproj/Mangrove_agb_AndamanAndNicobar_2000-01-01day_2009-12-31.tif (EPSG:4326)
[COGIFY] reproj/Mangrove_agb_AndamanAndNicobar_2000-01-01day_2009-12-31.tif → s3://ghgc-data-store-dev/transformed_cogs/CMS_Global_Map_Mangrove_Canopy/Mangrove_agb_AndamanAndNicobar_2000-01-01day_2009-12-31.tif


In [ ]:
# Save metadata if there are processed files
if len(files_processed) > 0:
    # Get metadata from one of the processed files
    sample_file = files_processed.iloc[0]['file_name']
    temp_sample_file = f"temp_{os.path.basename(sample_file)}"
    
    # Download sample file to extract metadata
    s3_client.download_file(raw_data_bucket, sample_file, temp_sample_file)
    
    with rasterio.open(temp_sample_file) as src:
        metadata = {
            "description": src.tags(),
            "driver": src.driver,
            "dtype": str(src.dtypes[0]),
            "nodata": src.nodata,
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "crs": str(src.crs),
            "transform": list(src.transform),
            "bounds": list(src.bounds),
            "total_files_processed": len(files_processed),
            "year": "2000"
        }
    
    # Upload metadata
    with tempfile.NamedTemporaryFile(mode="w+") as fp:
        json.dump(metadata, fp, indent=2)
        fp.flush()
        
        s3_client.upload_file(
            Filename=fp.name,
            Bucket=bucket_name,
            Key=f"{cog_data_prefix}/metadata.json",
        )
        print(f"Uploaded metadata to s3://{bucket_name}/{cog_data_prefix}/metadata.json")
    
    # Clean up sample file
    if os.path.exists(temp_sample_file):
        os.remove(temp_sample_file)

# Save the files_processed DataFrame to CSV using the same s3_client
with tempfile.NamedTemporaryFile(mode="w+", suffix=".csv") as fp:
    files_processed.to_csv(fp.name, index=False)
    fp.flush()
    
    s3_client.upload_file(
        Filename=fp.name,
        Bucket=bucket_name,
        Key=f"{cog_data_prefix}/files_converted.csv",
    )
    print(f"Saved processing log to s3://{bucket_name}/{cog_data_prefix}/files_converted.csv")

In [ ]:
# Display summary
print(f"\nProcessing Summary:")
print(f"Total files found: {len(keys)}")
print(f"Files processed: {len(files_processed)}")
print(f"\nProcessed files:")
files_processed


Processing Summary:
Total files found: 348
Files processed: 348

Processed files:


,file_name,COGs_created
0,coastal-observatory/data/Mangrove_agb_AndamanA...,Mangrove_agb_AndamanAndNicobar_2000-01-01day_2...
1,coastal-observatory/data/Mangrove_agb_Angola.tif,Mangrove_agb_Angola_2000-01-01day_2009-12-31.tif
2,coastal-observatory/data/Mangrove_agb_Anguilla...,Mangrove_agb_Anguilla_2000-01-01day_2009-12-31...
3,coastal-observatory/data/Mangrove_agb_AntiguaA...,Mangrove_agb_AntiguaAndBarbuda_2000-01-01day_2...
4,coastal-observatory/data/Mangrove_agb_Aruba.tif,Mangrove_agb_Aruba_2000-01-01day_2009-12-31.tif
...,...,...
343,coastal-observatory/data/Mangrove_hmax95_Venez...,Mangrove_hmax95_Venezuela_2000-01-01day_2009-1...
344,coastal-observatory/data/Mangrove_hmax95_Vietn...,Mangrove_hmax95_Vietnam_2000-01-01day_2009-12-...
345,coastal-observatory/data/Mangrove_hmax95_Virgi...,Mangrove_hmax95_VirginIslandsUs_2000-01-01day_...
346,coastal-observatory/data/Mangrove_hmax95_Walli...,Mangrove_hmax95_WallisAndFutuna_2000-01-01day_...
